# 1.	Cargar un dataset del Proyecto Sello

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion2-fundamentos-spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/04 14:05:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
ORIGEN_DATOS = "/opt/data"
ARTIFACTS = "/opt/artifacts"

Definición formal del esquema explícito acorde a la descripción técnica

In [3]:
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, BooleanType
schema_trades = StructType([
    StructField("trade_id", LongType(), True),
    StructField("price", DoubleType(), True),
    StructField("qty", DoubleType(), True),
    StructField("quote_qty", DoubleType(), True),
    StructField("time", LongType(), True),
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True)
])

Carga del CSV aplicando el esquema explícito sin inferencia de tipos

In [4]:
csv_path = f"{ORIGEN_DATOS}/BTCUSDT-trades-2026-01-05.csv"

df_raw = spark.read \
    .option("header", "false") \
    .schema(schema_trades) \
    .csv(csv_path)

Verificación de esquema contra columnas esperadas mediante teoría de conjuntos

In [5]:
expected_columns = {"trade_id", "price", "qty", "quote_qty", "time", "is_buyer_maker", "is_best_match"}
current_columns = set(df_raw.columns)

assert expected_columns == current_columns, f"Error en esquema: Faltan o sobran columnas. Obtenido: {current_columns}"

print("Esquema verificado correctamente:")
df_raw.printSchema()
df_raw.show(5, False)

Esquema verificado correctamente:
root
 |-- trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- qty: double (nullable = true)
 |-- quote_qty: double (nullable = true)
 |-- time: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)

+----------+--------+-------+-----------+----------------+--------------+-------------+
|trade_id  |price   |qty    |quote_qty  |time            |is_buyer_maker|is_best_match|
+----------+--------+-------+-----------+----------------+--------------+-------------+
|5734054604|91529.74|2.2E-4 |20.1365428 |1767571200308618|false         |true         |
|5734054605|91529.74|0.01   |915.2974   |1767571200375801|false         |true         |
|5734054606|91529.74|0.00437|399.9849638|1767571200477157|false         |true         |
|5734054607|91529.74|0.00764|699.2872136|1767571200480543|false         |true         |
|5734054608|91529.74|0.00136|124.4804464|1767571200545508|false   

# 2.	Filtrar y ordenar resultados

--- FILTRADO (Técnica 1: Sintaxis SQL Expresiva / col()) ---

Filtrar operaciones relevantes: Volumen en USDT mayor a 500 y ejecutadas como Maker

In [6]:
from pyspark.sql import functions as F

In [7]:
df_filtered_t1 = df_raw.filter(
    (F.col("quote_qty") > 500.0) & (F.col("is_buyer_maker") == True)
)

--- FILTRADO (Técnica 2: Expresión en Cadena estilo SQL) ---

Filtrar operaciones de volumen alto mediante condición literal

In [8]:
df_filtered_t2 = df_raw.filter("qty >= 0.01 AND is_best_match = true")

print(f"Registros filtrados (Técnica 1 - col): {df_filtered_t1.count()}")
print(f"Registros filtrados (Técnica 2 - SQL string): {df_filtered_t2.count()}")

Registros filtrados (Técnica 1 - col): 167796
Registros filtrados (Técnica 2 - SQL string): 264064


--- ORDENAMIENTO (Técnica 1: Columna explícita con control de nulos desc_nulls_last) ---

Ordenar por el timestamp de microsegundos de forma descendente

In [9]:
df_sorted_t1 = df_filtered_t1.sort(F.col("time").desc_nulls_last())

--- ORDENAMIENTO (Técnica 2: orderBy multinivel) ---

Ordenar prioritariamente por monto total operado descendente y luego por precio

In [10]:
df_sorted_t2 = df_filtered_t1.orderBy(F.col("quote_qty").desc(), F.col("price").asc())

print("\nMuestra de datos filtrados y ordenados (Técnica 2):")
df_sorted_t2.select("trade_id", "price", "qty", "quote_qty", "time").show(5)


Muestra de datos filtrados y ordenados (Técnica 2):
+----------+--------+--------+--------------+----------------+
|  trade_id|   price|     qty|     quote_qty|            time|
+----------+--------+--------+--------------+----------------+
|5734935723| 92530.0|18.22665|  1686511.9245|1767587199960033|
|5736967883|93438.47| 8.98579|839618.4693413|1767626517386309|
|5735369518| 92500.0| 6.44694|     596341.95|1767602487208792|
|5735369516| 92500.0| 6.32497|    585059.725|1767602487208694|
|5735400776| 92400.0| 5.76911|    533065.764|1767603716058771|
+----------+--------+--------+--------------+----------------+
only showing top 5 rows


# 3.	Confirmar o tratar duplicados

In [11]:
from pyspark.sql.window import Window

--- TÉCNICA 1: Uso de dropDuplicates() ---

In [12]:
df_dedup_t1 = df_raw.dropDuplicates(subset=["trade_id"])

--- TÉCNICA 2: Window Function + row_number() ---

Permite un control más avanzado al seleccionar cuál registro conservar según criterio

In [13]:
window_spec = Window.partitionBy("trade_id").orderBy(F.col("time").desc())

df_dedup_t2 = df_raw.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

Validación y comparación de resultados entre ambas técnicas

In [14]:
count_raw = df_raw.count()
count_dedup_1 = df_dedup_t1.count()
count_dedup_2 = df_dedup_t2.count()

print(f"Total registros originales: {count_raw}")
print(f"Total tras dropDuplicates(): {count_dedup_1}")
print(f"Total tras Window + row_number(): {count_dedup_2}")

[Stage 17:>                                                       (0 + 10) / 10]

Total registros originales: 5485489
Total tras dropDuplicates(): 5485489
Total tras Window + row_number(): 5485489


# 4.	Detectar y tratar nulos

Profiling y diagnóstico de nulos por columna

In [15]:
print("--- Diagnóstico de valores nulos ---")
null_counts = df_dedup_t1.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_dedup_t1.columns
])
null_counts.show()

--- Diagnóstico de valores nulos ---


[Stage 23:=====>                                                   (1 + 9) / 10]

+--------+-----+---+---------+----+--------------+-------------+
|trade_id|price|qty|quote_qty|time|is_buyer_maker|is_best_match|
+--------+-----+---+---------+----+--------------+-------------+
|       0|    0|  0|        0|   0|             0|            0|
+--------+-----+---+---------+----+--------------+-------------+



Tratamiento documentado según regla de negocio:

- Criterio para 'trade_id', 'price', 'qty': Si son nulos, la transacción no es válida para análisis financiero -> .na.drop()

- Criterio para banderas booleanas ('is_buyer_maker', 'is_best_match'): Si son nulas, se imputa un valor por defecto seguro (False) -> .na.fill()

In [16]:
df_clean = df_dedup_t1 \
    .na.drop(subset=["trade_id", "price", "qty"]) \
    .na.fill({
        "is_buyer_maker": False,
        "is_best_match": False,
        "quote_qty": 0.0
    })

print("Tratamiento de nulos completado. Registros finales limpios:", df_clean.count())

[Stage 29:=====>                                                   (1 + 9) / 10]

Tratamiento de nulos completado. Registros finales limpios: 5485489


# 5. Enriquecimiento de datos y escritura particionada en Parquet

In [17]:
from pyspark.sql.types import (
    StructType, StructField, LongType, DoubleType, BooleanType, TimestampType, StringType
)

Definir columna categórica relevante para particionar:

Transformamos el timestamp (microsegundos) a fecha (YYYY-MM-DD) y clasificamos la transacción por tipo de ejecutor (trade_type)

In [18]:
df_enriched = df_clean \
    .withColumn("event_timestamp", (F.col("time") / 1000000).cast(TimestampType())) \
    .withColumn("trade_date", F.to_date(F.col("event_timestamp"))) \
    .withColumn("trade_type", F.when(F.col("is_buyer_maker") == True, "MAKER_BUY").otherwise("TAKER_BUY"))

Control del número de archivos por partición usando repartition()

In [19]:
output_path = f"{ORIGEN_DATOS}/silver/binance_trades_parquet"

df_enriched \
    .repartition(1, "trade_type") \
    .write \
    .mode("overwrite") \
    .partitionBy("trade_type") \
    .parquet(output_path)

print(f"Datos guardados en Parquet particionados por 'trade_type' en: {output_path}")

[Stage 40:>                                                         (0 + 1) / 1]

Datos guardados en Parquet particionados por 'trade_type' en: /opt/data/silver/binance_trades_parquet


# 6. Lectura, verificación de PartitionFilters y análisis de balanceo

Leer de vuelta los datos desde la salida particionada

In [20]:
df_parquet_in = spark.read.parquet(output_path)

Filtrar por la columna de partición para gatillar la optimización de Partition Pruning

In [21]:
df_filtered_partition = df_parquet_in.filter(F.col("trade_type") == "MAKER_BUY")

In [22]:
print("--- Plan de Ejecución (Examine el filtro 'PartitionFilters') ---")
df_filtered_partition.explain(True)

--- Plan de Ejecución (Examine el filtro 'PartitionFilters') ---
== Parsed Logical Plan ==
'Filter '`=`('trade_type, MAKER_BUY)
+- Relation [trade_id#414L,price#415,qty#416,quote_qty#417,time#418L,is_buyer_maker#419,is_best_match#420,event_timestamp#421,trade_date#422,trade_type#423] parquet

== Analyzed Logical Plan ==
trade_id: bigint, price: double, qty: double, quote_qty: double, time: bigint, is_buyer_maker: boolean, is_best_match: boolean, event_timestamp: timestamp, trade_date: date, trade_type: string
Filter (trade_type#423 = MAKER_BUY)
+- Relation [trade_id#414L,price#415,qty#416,quote_qty#417,time#418L,is_buyer_maker#419,is_best_match#420,event_timestamp#421,trade_date#422,trade_type#423] parquet

== Optimized Logical Plan ==
Filter (isnotnull(trade_type#423) AND (trade_type#423 = MAKER_BUY))
+- Relation [trade_id#414L,price#415,qty#416,quote_qty#417,time#418L,is_buyer_maker#419,is_best_match#420,event_timestamp#421,trade_date#422,trade_type#423] parquet

== Physical Plan ==


Análisis de distribución y balanceo de la columna de partición

In [23]:
print("\n--- Distribución de registros por valor de la columna de partición ---")
partition_balance = df_parquet_in.groupBy("trade_type") \
    .agg(
        F.count("*").alias("total_records"),
        F.round(F.sum("quote_qty"), 2).alias("total_volume_usdt")
    )

partition_balance.show()


--- Distribución de registros por valor de la columna de partición ---
+----------+-------------+-----------------+
|trade_type|total_records|total_volume_usdt|
+----------+-------------+-----------------+
| MAKER_BUY|      2507141|   9.1159649162E8|
| TAKER_BUY|      2978348|  1.01585875193E9|
+----------+-------------+-----------------+



# 7. Mapeo conceptual del Pipeline en Arquitectura Medallón

In [24]:
"""
================================================================================
ARQUITECTURA MEDALLÓN (BRONZE, SILVER, GOLD) EN EL PIPELINE DEL PROYECTO SELLO
================================================================================

1. CAPA BRONZE (Raw Ingestion / Raw Data):
   - Descripción: Representa los datos ingestados en su estado original, sin modificaciones,
     preservando la fidelidad de la fuente.
   - Artefacto Real del Proyecto: 
     * Archivo fuente: 'BTCUSDT-trades-2026-01-05.csv'
     * DataFrame inicial cargado con esquema explícito: 'df_raw'

2. CAPA SILVER (Cleansed & Enriched Data):
   - Descripción: Datos limpios, deduplicados, validados contra nulos y enriquecidos con 
     tipos de datos adecuados y columnas derivadas para analítica.
   - Artefacto Real del Proyecto:
     * DataFrame procesado: 'df_enriched'
     * Almacenamiento optimizado: Parquet particionado en './data/silver/binance_trades_parquet'

3. CAPA GOLD (Business & Analytical Aggregations):
   - Descripción: Capa orientada a consumo de negocio, tableros de control o modelos ML. 
     Contiene métricas agregadas y datos altamente optimizados.
   - Artefacto Real del Proyecto:
     * DataFrame agregador: 'partition_balance' o métricas calculadas por ventana temporal (KPIs de volumen operado por 'trade_type').
"""
print("Definición e identificación de capas Bronze, Silver y Gold documentada.")

Definición e identificación de capas Bronze, Silver y Gold documentada.
